# Assignment 2: Warmtenet under uncertainty

**Mathematical Optimization (XM_0051), VU Amsterdam, 2026-27.** Individual assignment, **due Monday 7 December 2026, 23:59** (Canvas). Late submissions are not accepted.

### How this assignment works
- **Personal instance.** Setup uses your **student number** to generate data for the same district as in A1. Enter it *before starting the questions*.
- **Questions.** Parts 1–3 carry 90 points and *Verification and GenAI use* carries 10. Your grade is the total divided by 10. Part 2 builds on `build_sp` from 1b, and later comparisons use `evaluate` from 1d after the audit in 1f. You can start 3a–b independently. If a solve ends without an optimum, record its status and continue with parts that can proceed independently.
- **What to submit.** Fill in `ANSWERS`, choose *Restart and run all*, then save the notebook with its outputs and upload the `.ipynb` to Canvas. Staff check your numbers and decisions against your instance and mark explanations by hand.
- **Course code.** You may adapt code from the course labs and worked solutions. Cite the exercise or cell and explain how you used it.
- **Runtime.** Our local runs took about a minute, excluding Setup. Times depend on the instance and hardware. If a solve is slow, check the formulation size, bounds and solver status.
- **Generative AI** (VU Level 3) may be used in your work, provided that you explain its use and critically assess its output in the last section. You are responsible for every model, line of code, number and explanation you submit.

## Setup
Open the Canvas notebook in Colab, save a copy in your Google Drive and run the Setup cells below. The first installs the assignment package with its solver and other dependencies, which can take a minute or more on a new runtime.

For local Jupyter, use Python 3.10–3.14. Install the package in the notebook's environment with `python -m pip install --upgrade "mo2026 @ git+https://github.com/alessandrozocca/MO2026@a2-v0.2.2"`. Run Setup first in each runtime. If you imported the A1 package in this session, install A2, restart the session and rerun Setup.

In [ ]:
#@title Install assignment packages
import sys

if "google.colab" in sys.modules:
    %pip install -q --upgrade "mo2026 @ git+https://github.com/alessandrozocca/MO2026@a2-v0.2.2"

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo

from mo2026.common import solve, relax_integrality, check_answer_format, print_answer_block
from mo2026.warmtenet_a2 import make_instance_a2, one_source_example, plot_network_a2, winters

**Enter your student number** in digits, followed by the same suffix if staff assigned one in A1. Keep that identifier throughout A2 because all data depend on it.

In [ ]:
STUDENT_NUMBER = None  # e.g. 2712345, or "2712345-1" if staff assigned that suffix

assert STUDENT_NUMBER is not None, "Enter your student number first."

For each solve, call `solve(model)`. It runs HiGHS, loads any available solution into the Pyomo model and returns a dictionary. Check `status` before using objective or variable values, since a time limit can leave you with a feasible solution whose optimality is unproven. See `help(solve)` for fields such as the best bound and relative MIP gap.

## The story so far

Grachtenwarmte built Tweegrachten's heat network with the technology from A1 Q1. The Q4 extensions apply only to A1. The supplied design connects as many neighborhoods as possible, then minimizes cost. If all can connect, this is Q1 with every $z_k=1$. The connected neighborhoods form `inst.K`, and construction costs are sunk.

**New housing** has raised design-day demand by 30–60% in four neighborhoods (`inst.growth`). Changes in the network, demands and included costs also affect any cost comparison with A1.

> Which capacities and pipe investments should Grachtenwarmte choose before the winter, and how should it weigh cost against reliability?

The figure shows each winter's coldest daily mean temperature in KNMI's homogenized De Bilt series, with winter 1951 running from November 1950 to March 1951. The *design day* has a mean of −10 °C. The network, costs and demands are synthetic. The package's [data README](https://github.com/alessandrozocca/MO2026/blob/a2-v0.2.2/mo2026/data/README.md) documents the observed temperatures.

In [ ]:
inst = make_instance_a2(STUDENT_NUMBER)
print(f"Warmtenet-A2 instance with {len(inst.V)} nodes, {len(inst.E_old)} existing pipes, "
      f"{len(inst.E_new)} candidate new pipes, {len(inst.S)} sources, "
      f"{len(inst.K)} connected neighborhoods, and {len(inst.train_demand)} training winters.")
w = winters()
ax = w["coldest_day_mean_C"].plot(marker="o", figsize=(9, 3), title="Coldest daily mean temperature per winter, De Bilt (KNMI)")
ax.axhline(-10, color="k", ls="--", lw=1, label="design day (-10 °C)")
ax.axvspan(1950.5, 1995.5, color="0.9", label="training winters")
ax.set_ylabel("°C"); ax.legend(); plt.show()
plot_network_a2(inst)
plt.show()

### Data and units

Costs are in **k€/yr** and demand, output, flows and capacities in **MW** at the winter peak. Operating coefficients convert peak-day quantities to annual costs. For example, winter production cost is proportional to peak output.

| Symbol | Python | Meaning |
|---|---|---|
| $\kappa_s$ | `inst.reservation_cost[s]` | reservation cost per MW at source $s$ |
| $c_s$ | `inst.heat_cost[s]` | heat cost per MW at source $s$ |
| $t_e$ | `inst.transport_cost[e]` | transport cost per MW of flow on pipe $e$ (as in A1) |
| $\bar C_s$ | `inst.max_capacity[s]` | maximum reservation at source $s$ (MW) |
| $\pi$ | `inst.local_capacity_cost` | local boiler capacity cost per MW |
| $c_\ell$ | `inst.local_heat_cost` | local boiler heat cost per MW |
| $\bar q_k$ | `inst.local_max[k]` | maximum local capacity at neighborhood $k$ (MW) |
| $p$ | `inst.backup_cost` | mobile-boiler cost per MW |
| $\nu_e$ | `inst.upgrade_cost[e]` | cost of upgrading existing pipe $e$ |
| $\Delta_e$ | `inst.upgrade_capacity[e]` | extra capacity from upgrading pipe $e$ (MW) |
| $f_e$ | `inst.build_cost[e]` | cost of building new pipe $e$ |
| $C_e$ | `inst.cap[e]` | base capacity of pipe $e$ (MW) |
| $D_k$ | `inst.design_demand[k]` | design-day demand with housing growth, before the demand factor (MW) |
| $\beta_k$ | `inst.base_share[k]` | temperature-independent share of $D_k$ |

`inst.E` combines `inst.E_old` and `inst.E_new`. The arcs entering and leaving node $i$ are `inst.in_arcs[i]` and `inst.out_arcs[i]`.

**First-stage decisions, before the winter.** Grachtenwarmte chooses capacities and pipes in summer.

- $r_s \in [0, \bar C_s]$, capacity reserved at source $s\in S$
- $q_k \in [0, \bar q_k]$, local peak-boiler capacity at neighborhood $k\in K$
- $\text{up}_e$, binary decision to double the capacity of existing pipe $e\in E_{\text{old}}$ with a parallel pipe
- $\text{new}_e$, binary decision to build candidate pipe $e\in E_{\text{new}}$

**Second-stage dispatch, on the coldest day.** These recourse decisions depend on the realized demands.

- $x_{ij}$, heat flow on arc $(i,j)$
- $g_s \le r_s$, source output
- $\ell_k \le q_k$, local boiler output
- $b_k \ge 0$, output from rented mobile boilers

Mobile boilers are always available at high cost (`backup` in the code and `ANSWERS`). Local and mobile boilers supply only their own neighborhood, so $\ell_k+b_k\le d_k$.

**Demand model.** In a winter whose coldest day has mean temperature $T$, the peak demand of neighborhood $k$ is
$$
d_k = D_k\left(\beta_k + (1-\beta_k)\,\frac{18 - T}{18 - (-10)}\right)\cdot \varepsilon_k.
$$
The share $\beta_k$ accounts for uses such as hot water. The small random factor $\varepsilon_k$ represents neighborhood-specific effects such as occupancy or wind.

The 45 equally likely **training winters** (1951–1995) index the rows of `inst.train_demand`, with one column per neighborhood. The plot shows 31 later winters for context.

Part 2 uses $\varepsilon$ for the allowed violation probability, distinct from the demand factor $\varepsilon_k$.

In [ ]:
display(inst.train_demand.head())
display(pd.DataFrame({"design demand": inst.design_demand, "growth": inst.growth, "base share": inst.base_share}))
display(pd.DataFrame({"name": inst.source_name, "max capacity": inst.max_capacity,
                      "reservation cost": inst.reservation_cost, "heat cost": inst.heat_cost}))

### The model for a single winter (given)
Before building the scenario model, read `build_winter_model`. It optimizes investments and dispatch together for one winter's peak `demand` dictionary $k \mapsto d_k$.

We call the first-stage decisions a *plan*. `first_stage_of(m, inst)` returns a full-precision dictionary and `show_plan` a rounded copy for display. `fix_first_stage(m, inst, fs)` fixes the plan in another model with the same first-stage variables and makes those binaries continuous. The result is an LP if no other integer variables, such as scenario indicators, remain.

In [ ]:
def build_winter_model(inst, demand):
    m = pyo.ConcreteModel()
    out_arcs = {i: [a for a in inst.A if a[0] == i] for i in inst.V}
    in_arcs = {i: [a for a in inst.A if a[1] == i] for i in inst.V}
    # First-stage decisions
    m.r = pyo.Var(inst.S, bounds=lambda m, s: (0, inst.max_capacity[s]))   # reserved source capacity (MW)
    m.q = pyo.Var(inst.K, bounds=lambda m, k: (0, inst.local_max[k]))      # local peak-boiler capacity (MW)
    m.up = pyo.Var(inst.E_old, domain=pyo.Binary)                           # upgrade an existing pipe
    m.new = pyo.Var(inst.E_new, domain=pyo.Binary)                          # build a new pipe
    # Second-stage dispatch
    m.x = pyo.Var(inst.A, domain=pyo.NonNegativeReals)   # heat flow (MW)
    m.g = pyo.Var(inst.S, domain=pyo.NonNegativeReals)   # production at the sources (MW)
    m.l = pyo.Var(inst.K, domain=pyo.NonNegativeReals)   # production of the local boilers (MW)
    m.b = pyo.Var(inst.K, domain=pyo.NonNegativeReals)   # mobile-boiler output (MW)

    m.first_stage_cost = pyo.Expression(expr=
        sum(inst.reservation_cost[s] * m.r[s] for s in inst.S)
        + sum(inst.local_capacity_cost * m.q[k] for k in inst.K)
        + sum(inst.upgrade_cost[e] * m.up[e] for e in inst.E_old)
        + sum(inst.build_cost[e] * m.new[e] for e in inst.E_new))
    m.operation_cost = pyo.Expression(expr=
        sum(inst.heat_cost[s] * m.g[s] for s in inst.S)
        + sum(inst.transport_cost[inst.edge(i, j)] * m.x[i, j] for (i, j) in inst.A)
        + sum(inst.local_heat_cost * m.l[k] + inst.backup_cost * m.b[k] for k in inst.K))
    m.cost = pyo.Objective(expr=m.first_stage_cost + m.operation_cost)

    @m.Constraint(inst.V)
    def balance(m, i):
        inflow = sum(m.x[a] for a in in_arcs[i]) - sum(m.x[a] for a in out_arcs[i])
        supply = m.g[i] if i in inst.S else 0
        need = (demand[i] - m.l[i] - m.b[i]) if i in inst.K else 0
        return inflow + supply == need

    @m.Constraint(inst.E)
    def pipe(m, i, j):
        if (i, j) in inst.upgrade_cost:
            return m.x[i, j] + m.x[j, i] <= inst.cap[i, j] + inst.upgrade_capacity[i, j] * m.up[i, j]
        return m.x[i, j] + m.x[j, i] <= inst.cap[i, j] * m.new[i, j]

    m.source = pyo.Constraint(inst.S, rule=lambda m, s: m.g[s] <= m.r[s])
    m.local = pyo.Constraint(inst.K, rule=lambda m, k: m.l[k] <= m.q[k])
    m.own = pyo.Constraint(inst.K, rule=lambda m, k: m.l[k] + m.b[k] <= demand[k])  # boilers serve only k
    return m


def first_stage_of(m, inst):
    # Rounding a capacity down can make the fixed plan infeasible in another model.
    return {"reserve": {s: pyo.value(m.r[s]) for s in inst.S},
            "local": {k: pyo.value(m.q[k]) for k in inst.K},
            "upgrade": sorted(e for e in inst.E_old if pyo.value(m.up[e]) > 0.5),
            "new": sorted(e for e in inst.E_new if pyo.value(m.new[e]) > 0.5)}


def show_plan(fs, digits=3):
    # Return a rounded copy for display.
    return {"reserve": {s: round(v, digits) for s, v in fs["reserve"].items()},
            "local": {k: round(v, digits) for k, v in fs["local"].items() if abs(v) > 10 ** -digits},
            "upgrade": fs["upgrade"], "new": fs["new"]}


def fix_first_stage(m, inst, fs):
    def fix_capacity(var, value):
        # Keep full precision and suppress bound warnings within 1e-8 MW.
        near_bounds = ((var.lb is None or value >= var.lb - 1e-8)
                       and (var.ub is None or value <= var.ub + 1e-8))
        var.fix(value, skip_validation=near_bounds)

    for s in inst.S:
        fix_capacity(m.r[s], fs["reserve"][s])
    for k in inst.K:
        fix_capacity(m.q[k], fs["local"][k])
    for e in inst.E_old:
        m.up[e].fix(1 if tuple(e) in set(map(tuple, fs["upgrade"])) else 0)
        m.up[e].domain = pyo.UnitInterval   # Drop the integer domain to allow LP solves.
    for e in inst.E_new:
        m.new[e].fix(1 if tuple(e) in set(map(tuple, fs["new"])) else 0)
        m.new[e].domain = pyo.UnitInterval
    return m


# Optimize for the design-day demand at -10 °C, with the demand factor set to one.
m0 = build_winter_model(inst, inst.demand_at(-10))
print(solve(m0)["objective"], show_plan(first_stage_of(m0, inst)))

### Reporting helpers
`reliability_report`, `capacity_report` and `comparison_table` format results. `first_stage_cost` returns a plan's first-stage cost.

Keep results and plans at full precision for later calculations and `ANSWERS`, and round only for display. Reuse solves and evaluations with the same inputs.

In [ ]:
def reliability_report(rows, rp):
    columns = ["eps", "status", "allowed training winters", "cost", "training winters with mobile boilers"]
    table = pd.DataFrame(rows, columns=columns).set_index("eps")
    _, ax = plt.subplots()
    successful = table["cost"].notna()
    ax.plot(table.index[successful], table.loc[successful, "cost"], marker="o", label="cost")
    ax.set_title("Price of reliability")
    ax.axhline(rp, ls="--", color="k", label="RP")
    values = [rp] + table["cost"].dropna().tolist()
    lower, upper = min(values), max(values)
    margin = 0.05 * max(upper - lower, 1.0)
    ax.set_ylim(lower - margin, upper + margin)
    ax.set_xlabel("epsilon"); ax.set_ylabel("expected annual cost (k€/yr)")
    ax.legend()
    return table, ax


def capacity_report(rows, tol=1e-6):
    table = pd.DataFrame(rows, columns=["constraint", "worst-case load", "capacity"])
    table["residual"] = table["capacity"] - table["worst-case load"]
    table["violated"] = table["residual"] < -tol
    table["tight"] = table["residual"].abs() <= tol
    return table


def comparison_table(plan_names):
    return pd.DataFrame(index=list(plan_names), dtype=float, columns=[
        "first-stage cost", "expected annual cost", "training winters with mobile boilers", "90% CVaR"])


def first_stage_cost(inst, fs):
    return (sum(inst.reservation_cost[s] * fs["reserve"][s] for s in inst.S)
            + sum(inst.local_capacity_cost * fs["local"][k] for k in inst.K)
            + sum(inst.upgrade_cost[tuple(e)] for e in fs["upgrade"])
            + sum(inst.build_cost[tuple(e)] for e in fs["new"]))

---
## Part 1. A two-stage stochastic program (35 points)

Choose a plan that minimizes expected annual cost across the training winters and compare it with simpler benchmarks. Lecture 8 and tutorial sheet 4 Exercise 3 cover scenario models and their benchmarks. Computer lab W4 Exercise 1(f) gives a scaffold for fixed-plan evaluation.

**1a [6 pt].** Formulate a **two-stage stochastic program (SP)** with the $N = 45$ training winters, each with probability $1/N$. State the first-stage variables and cost. Write the second-stage problem for demand vector $d(\omega)$ and define its optimal value, the *recourse function*. Then write the **extensive form**.

Explain why the first-stage decisions must be the same in every winter (nonanticipativity) and why dispatch may depend on the winter. Justify relatively complete recourse by showing that every admissible first-stage plan has a feasible dispatch in every training winter.

*Write your answer here.*

**1b [6 pt].** Implement the extensive form as `build_sp(inst, demand)`. The rows of `demand` index winters `W = list(demand.index)`, with entries `demand.at[w, k]`. `first_stage_of` expects the first-stage variables `r, q, up, new`. For the audit in 1f, put the winter index first on dispatch variables `x, g, l, b`, as in `m.b[w, k]`.

Solve for the 45 training winters with `solve(m, options={"mip_rel_gap": 0.0})`. HiGHS's default relative MIP gap tolerance is $10^{-4}$, so `optimal` can be reported with a positive gap. Set that tolerance to zero here and in 2b for the cost comparison.

Report **RP**, the recourse problem's optimal expected annual cost. Plot the plan with `plot_network_a2(inst, upgraded=..., new_pipes=..., reservation=...)` and display it with `show_plan`, including local boiler capacities. Keep the plan for later solves and `ANSWERS`.

In [ ]:
def build_sp(inst, demand):
    m = pyo.ConcreteModel()
    # Use r, q, up, new for the first stage and x, g, l, b with the winter index first for dispatch.
    # Add your extensive form here.
    return m

**1c [6 pt] Verification with a one-source example.** A single source with reservation cost $\kappa$ and heat cost $c$ serves demand without a network. Mobile boilers at cost $p$ cover any shortfall. The data below give $D_\omega$, the sum of neighborhood demands in winter $\omega$.
1. Your data satisfy $0 < \kappa < p-c$. Show that an optimal reservation is the smallest value $r$ among the $D_\omega$ for which the fraction of winters with $D_\omega > r$ is at most $\kappa/(p-c)$, and compute it. *Hint:* what is the marginal effect of reserving one more MW?
2. Solve the same example as an extensive form with $N$ winters. Check that the LP reservation equals your closed-form value or, if the cost is flat between two consecutive ordered demands, that it lies in that interval and gives the same cost.
3. Relate this example to Lecture 7's seafood (newsvendor) example. Which quantity plays the role of the critical ratio, and which costs does $\kappa/(p-c)$ compare? Computer lab W4 Exercise 2 practices the newsvendor model.

In [ ]:
ex = one_source_example(STUDENT_NUMBER)
D = pd.Series(ex["demand"])
print({k: v for k, v in ex.items() if k != "demand"}, "| N =", len(D))

In [ ]:
# Write your (commented) code here.

*Write your answer here.*

**1d [7 pt] EVPI and VSS.** Write `evaluate(inst, fs, demand)` to fix the plan `fs` and minimize dispatch cost for each winter in `demand`. Return the average annual cost, including first-stage cost, and the number of winters with total mobile-boiler output above $10^{-6}$ MW. *Hint:* build the extensive form and use `fix_first_stage` with the dictionary from `first_stage_of`.

- The **wait-and-see value (WS)** averages the optimal costs for the separate training winters, each solved with the full demand vector known before investing.
- The **EV solution** comes from solving `build_winter_model` at mean training demand. Call `evaluate` on that plan to obtain **EEV**, its expected cost over the training winters.

With **RP** from 1b, compute $\text{EVPI} = \text{RP} - \text{WS}$ and $\text{VSS} = \text{EEV} - \text{RP}$. Lecture 8's *Benchmark conventions: profit and cost* gives these cost formulas and reverses the subtraction for the farmer's profits. Both gaps are nonnegative under either convention.

Interpret EVPI and VSS in words and in k€/yr, and give VSS as a percentage of RP. Report the number of training winters with mobile boilers for the RP and EV plans for use in 2b and 3e.

In [ ]:
def evaluate(inst, fs, demand):
    # Return (average annual cost, number of winters with total mobile-boiler output > 1e-6 MW).
    pass

*Write your answer here.*

**1e [6 pt] The upper tail of annual cost.** Fix the RP and EV plans from 1b and 1d and use cost-minimizing dispatch over the same 45 equally likely winters. The scenario loss $L_\omega$ is the total annual cost in winter $\omega$, including first-stage cost once.
1. Compute the scenario losses and check that their means reproduce RP and EEV.
2. Compute **Value-at-Risk** $\operatorname{VaR}_{0.9}(L)$ as the smallest loss with cumulative probability at least 0.9 (the generalized quantile). Default `np.quantile` interpolation can give a different value. For **Conditional Value-at-Risk** $\operatorname{CVaR}_{0.9}(L)$, implement Lecture 8's LP in the threshold and excess losses with fixed scenario losses. Averaging the five largest losses can differ from CVaR because the worst 10% has the mass of 4.5 winters. Report mean cost, VaR and CVaR for each plan.
3. Rank RP and EV by expected cost and CVaR, allowing ties. Explain whether these rankings must agree using Lecture 8's *Farmer: expected profit versus CVaR* at level $1/2$. Computer lab W4 Exercise 1(g) practices this comparison.

*Hint:* fix each plan and solve for dispatch once. Add `first_stage_cost(inst, fs)` to each winter's operating cost. Tutorial sheet 4 Exercise 8 uses the same loss convention.

In [ ]:
# Write your (commented) code here.

*Write your answer here.*

**1f [4 pt] Audit an evaluator.** A colleague obtained `evaluate_chatbot` from a chatbot, and it reproduces the RP objective for the RP plan.
1. Before calling the function on `zero_plan`, derive that plan's expected annual cost and number of winters with mobile boilers directly from the demand data.
2. Find the **two errors** and explain their effects. Why does the faulty function still reproduce the RP objective? What does that check establish?
3. Give a targeted test for each error that fails on the colleague's function, isolating the errors if needed. Leave the supplied `evaluate_chatbot` unchanged.
4. Run both tests on your `evaluate` from 1d. If either fails, correct `evaluate` **in the original 1d cell**, rerun the notebook and describe the change here. Report the resulting evaluations of the RP plan and zero plan, and enter the zero-plan cost and winter count in `ANSWERS`.

In [ ]:
def evaluate_chatbot(inst, fs, demand):
    m = build_sp(inst, demand)
    for s in inst.S:
        m.r[s].fix(fs["reserve"][s])
    for e in inst.E_old:
        m.up[e].fix(int(e in set(map(tuple, fs["upgrade"]))))
    for e in inst.E_new:
        m.new[e].fix(int(e in set(map(tuple, fs["new"]))))
    res = solve(m)
    count = sum(pyo.value(m.b[w, k]) > 1e-6 for w in demand.index for k in inst.K)
    return res["objective"], count


zero_plan = {"reserve": {s: 0.0 for s in inst.S}, "local": {k: 0.0 for k in inst.K},
             "upgrade": [], "new": []}

In [ ]:
# Write your (commented) code here.

*Write your answer here.*

---
## Part 2. Reliability: chance constraints (20 points)

Grachtenwarmte's board wants to limit mobile boilers in residential streets to at most a fraction $\varepsilon$ of winters. **CC($\varepsilon$)** is the joint chance-constrained model with this allowed violation probability under the training distribution. Computer lab W4 Exercise 4 compares joint and individual chance constraints.

**2a [7 pt] Joint chance constraint and LP relaxations.** Under the training distribution, require $\mathbb P(b_k(\omega)=0\ \forall k\in K)\ge1-\varepsilon$. Reformulate this constraint using one binary indicator $u_\omega$ per winter and tight valid big-$M$ bounds. Explain what $u_\omega=1$ allows and how mobile-boiler output is charged. See Lecture 7's *Finite scenarios: joint reliability* and *Finite scenarios: valid big-M bounds*.

Compare the LP relaxations at $\varepsilon=0.1$.

1. Predict the order of LP optima for SP, CC with tight bounds and CC with $M=10^6$, including whether you expect any inequality to be strict.
2. In the 2b code cell, use `relax_integrality` on fresh SP and tight CC models for the two LP solves. Report both optima and their difference. An absolute difference up to $10^{-3}$ k€/yr counts as numerical agreement.
3. Find the smallest relaxed $u_\omega$ allowed by the tight bounds for your SP LP dispatch in each winter. Compare their sum with $\lfloor\varepsilon N\rfloor=4$ and explain what it tells you about the two LP values. Check from the demand table whether $u_\omega=\max_k d_k(\omega)/10^6$ extends any feasible SP LP solution to the LP relaxation of CC with $M=10^6$.

*Write your answer here.*

**2b [6 pt] Price of reliability.** Use the RP plan's count of training winters with mobile boilers to give a sufficient condition for the CC optimum to equal RP. Record your prediction for each $\varepsilon \in \{0,\ 0.05,\ 0.1,\ 0.2\}$ before solving.

You may use the **last-resort property**. With these costs, if a fixed plan can serve a winter without mobile boilers, its cost-minimizing dispatch uses none.

Solve with the relative MIP gap setting from 1b. Tabulate each solve status and **winter allowance** $\lfloor\varepsilon N\rfloor$ (the violation budget in Lecture 7's *Interpreting a finite-scenario reliability budget*). For each optimal plan give the CC objective and the number of winters with mobile boilers from `evaluate`.

Plot cost against $\varepsilon$ with `reliability_report(rows, rp)`. Explain the *reliability premium* (the CC objective minus RP) and how investments respond to $\varepsilon$. If the pipes stay the same, examine reservations and local capacity. If a predicted equality seems to fail, check whether the RP dispatch fits the allowance and what the best bound and gap from `solve` show. Keep the CC(0.1) plan for 2c and 3e.

In [ ]:
def build_cc(inst, demand, eps):
    m = build_sp(inst, demand)
    # Add your joint chance constraint here.
    return m

# For 2a, solve the two LP relaxations before the integer models.
# sp_lp = relax_integrality(build_sp(inst, inst.train_demand))
# tight_lp = relax_integrality(build_cc(inst, inst.train_demand, 0.1))
# Record both solve statuses, the LP values and the checks on the relaxed indicators.

cc_rows = []
# For each epsilon, use the same gap setting as in 1b.
# res = solve(m, options={"mip_rel_gap": 0.0})
# Check the status and append a row with these keys:
# {"eps": ..., "status": ..., "allowed training winters": ...,
#  "cost": ..., "training winters with mobile boilers": ...}
# Use None for unavailable costs and counts. Keep the solver status.
# cc_table, ax = reliability_report(cc_rows, rp)
# display(cc_table)
# plt.show()

*Write your answer here.*

**2c [7 pt] Joint versus individual chance constraints.** See tutorial sheet 4 Exercises 4(d) and 5.
1. On paper, use binary indicators to formulate individual chance constraints with a common limit $\varepsilon$ on each neighborhood's mobile-boiler frequency.
2. Fix the CC(0.1) plan in a fresh `build_sp` model, solve and keep the dispatch. If CC(0.1) is unavailable, give its status.

   - **Frequencies.** For each neighborhood, mark winters in which `m.b[w, k]` exceeds $10^{-6}$ MW. Report each neighborhood's frequency, the largest individual frequency and the joint frequency (the fraction of winters marked in at least one neighborhood). In 1d the threshold applies to total mobile-boiler output in a winter.
   - **Dispatch check.** Trace a winter–neighborhood pair above the mobile-output threshold, or a peak-demand pair if none qualifies. Report its demand and verify that net pipe inflow plus local and mobile output add up to it. Pick a relevant pipe, source or local boiler from the map, then compute and interpret its slack (fixed capacity minus load). A slack within $10^{-6}$ MW of zero counts as binding.

3. Use the union bound to split the joint probability limit 0.1 equally among the neighborhoods. Report the common $\varepsilon_{\text{ind}}$ and integer winter allowance per neighborhood. Compare the sum of individual frequencies with the joint frequency using the marked winters. Does meeting each individual limit at 0.1 imply the joint limit at 0.1?

In [ ]:
# Keep the solved model for the frequencies and the dispatch check.
# Use inst.in_arcs[k] and inst.out_arcs[k] to compute net inflow at neighborhood k.

*Write your answer here.*

---
## Part 3. A robust plan (35 points)

The finance director is reluctant to rely on probabilities estimated from 45 winters. She wants a plan that serves every demand vector in an uncertainty set without mobile boilers. Before the winter, choose the investments and the proportional routing coefficients that define the **prescribed policy**.

Lecture 9 introduces uncertainty sets and the steps of robust optimization. For robust counterparts, see Lecture 10's slides on the *Budget ($\Gamma$) Uncertainty Set*, tutorial sheet 5 Exercises 4 and 6(b), and computer lab W5 Exercise 3(a–b).

Neighborhood $k$ receives fraction $\alpha_k$ of its demand from its local boiler. It has its own commodity, as in A1 Q2b. Here $\varphi^k_{ij} \ge 0$ is the flow on arc $(i,j)$ per unit of $k$'s demand and $\gamma_{ks}$ is the fraction of $k$'s demand produced at source $s$. The routing constraints are
$$
\sum_{(j,i)\in A}\varphi^k_{ji} - \sum_{(i,j)\in A}\varphi^k_{ij} + \gamma_{ki}[i\in S] = (1-\alpha_k)[i=k] \qquad \forall k\in K,\ i \in V ,
$$
where $[\cdot]$ is 1 if the condition holds and 0 otherwise. The flows $d_k\varphi^k$ and the associated source and local-boiler outputs must respect their capacities for every coldest-day demand vector in the **budget uncertainty set**
$$
U(\Gamma) = \Big\{ d : d_k = \bar d_k + \hat d_k\,\zeta_k,\ \ 0 \le \zeta_k \le 1,\ \ \sum_{k\in K}\zeta_k \le \Gamma \Big\}.
$$
Here $\bar d_k$ and $\bar d_k+\hat d_k$ are the mean and maximum of neighborhood $k$'s training demands. The budget $\Gamma\ge0$ bounds the sum of upward normalized deviations, so at $\Gamma=1$ two neighborhoods can each be halfway between mean and maximum. For integer $\Gamma$, the normalized set's vertices are 0–1 vectors with at most $\Gamma$ ones. This is the vertex interpretation of Lecture 10's “at most $\Gamma$ deviations.” Tutorial sheet 5 Exercise 6(b) gives the symmetric set, with a bound on the sum of absolute normalized deviations.

The robust optimization model **RO($\Gamma$)** minimizes first-stage cost plus the prescribed policy's operating cost at $\bar d$. In 3c and 3e, `evaluate` gives **flexible dispatch** by reoptimizing each winter with investments fixed.

In [ ]:
dbar = inst.train_demand.mean()
dhat = inst.train_demand.max() - dbar
pd.DataFrame({"dbar": dbar, "dhat": dhat}).round(2)

**3a [8 pt] Robust counterpart.** Consider a capacity constraint $\sum_{k} d_k\, a_k \le c$ that must hold for **all** $d \in U(\Gamma)$. The load per unit of neighborhood $k$'s demand is $a_k \ge 0$, and $c$ is the capacity. Both can depend on the decisions.
1. Write the adversarial LP in $\zeta$ for fixed decisions. Explain why strong LP duality applies and use it to show that the constraint holds for all $d\in U(\Gamma)$ exactly when some $\lambda,\mu_k\ge0$ satisfy

   $$
   \sum_k \bar d_k a_k + \Gamma \lambda + \sum_k \mu_k \le c, \qquad \lambda + \mu_k \ge \hat d_k\, a_k \quad \forall k .
   $$

   Here $\lambda$ multiplies the budget constraint and $\mu_k$ multiplies $\zeta_k\le1$. In Lecture 10's derivation with auxiliary variables, the nonpositive component $\mu_j^i$ multiplies $\tilde z_j\ge z_j$. You may also apply the polyhedral counterpart with the deterministic capacity expression moved to the left.
2. Write the full robust model with its first-stage and mean-demand operating costs, the routing constraints and the robust counterparts for pipes, sources and local boilers. Give the coefficients $a_k$ and capacity $c$ for each family.
3. The derivation in item 1 works for any sign of $a_k$. What does $a_k \ge 0$ imply for the load when a demand is below its mean? Simplify the robust local-boiler constraints and explain why they are independent of $\Gamma$ once $\Gamma\ge1$.

*Write your answer here.*

### Worked example: one capacity constraint

Each entry of `m.rows` labels one capacity constraint, or *row*. Order its nonnegative coefficients of $\zeta_k$, the *exposures* $h_k=\hat d_k a_k$, as $h_{(1)}\ge\cdots\ge h_{(|K|)}$. At $\Gamma=t+\theta<|K|$, with integer $t$ and $0\le\theta<1$, the largest deviation load is $\sum_{j=1}^t h_{(j)}+\theta h_{(t+1)}$. For $\Gamma\ge|K|$ it is $\sum_k h_k$.

Suppose one pipe carries the full demand of three neighborhoods, so $a=(1,1,1)$, with $\bar d=(4,5,6)$ and $\hat d=(3,2,1)$. Its nominal load is 15 MW. At $\Gamma=1.5$, the adversary spends one unit of budget on the first deviation and half on the second, giving $15+3+\tfrac12\cdot2=19$ MW.

The choice $\lambda=2$, $\mu=(1,0,0)$ satisfies $\lambda+\mu_k\ge h_k$ ($3\ge3$, $2\ge2$, $2\ge1$). The load bound $15+1.5\cdot2+1=19$ MW is attained by the deviations above, so the pair is an *optimal dual certificate*.

**3b [6 pt] Robust costs by budget.** Add the capacity constraints from 3a to `build_ro(inst, gamma, dbar, dhat)`, keeping the first-stage names `r, q, up, new`. The routing constraints are supplied.

Solve for $\Gamma \in \{0, 1, 2, 3, 4, 6, 8, |K|\}$, plot the optimal cost and describe the curve, including any flat stretches. Interpret the sets at $\Gamma=0$ and $\Gamma=|K|$, then show that $U(\Gamma_1)\subseteq U(\Gamma_2)$ whenever $\Gamma_1\le\Gamma_2$ and deduce that the optimal cost is nondecreasing.

Keep the solved models and full-precision plans. For 3c, retain the demand coefficients and capacities from the $\Gamma=2$ model. You can define functions so `m.a(m, row, k)` returns the coefficient of $d_k$ and `m.c(m, row)` the capacity for `m.rows[row]`.

**Optional checks.** Compare $\Gamma=0$ with the mean-demand model **without mobile boilers**. If the EV optimum from 1d uses no mobile boilers, its value should agree too. For $\Gamma=|K|$, apply the policy to the training winters and compare its capacity residuals with 3a.

In [ ]:
def build_ro(inst, gamma, dbar, dhat):
    m = pyo.ConcreteModel()
    K = inst.K
    out_arcs = {i: [a for a in inst.A if a[0] == i] for i in inst.V}
    in_arcs = {i: [a for a in inst.A if a[1] == i] for i in inst.V}
    m.r = pyo.Var(inst.S, bounds=lambda m, s: (0, inst.max_capacity[s]))
    m.q = pyo.Var(K, bounds=lambda m, k: (0, inst.local_max[k]))
    m.up = pyo.Var(inst.E_old, domain=pyo.Binary)
    m.new = pyo.Var(inst.E_new, domain=pyo.Binary)
    m.phi = pyo.Var(K, inst.A, domain=pyo.NonNegativeReals)
    m.gam = pyo.Var(K, inst.S, domain=pyo.NonNegativeReals)
    m.alpha = pyo.Var(K, bounds=(0, 1))
    first = (sum(inst.reservation_cost[s] * m.r[s] for s in inst.S) + sum(inst.local_capacity_cost * m.q[k] for k in K)
             + sum(inst.upgrade_cost[e] * m.up[e] for e in inst.E_old) + sum(inst.build_cost[e] * m.new[e] for e in inst.E_new))
    operation = sum(dbar[k] * (sum(inst.heat_cost[s] * m.gam[k, s] for s in inst.S)
                               + sum(inst.transport_cost[inst.edge(i, j)] * m.phi[k, i, j] for (i, j) in inst.A)
                               + inst.local_heat_cost * m.alpha[k]) for k in K)
    m.cost = pyo.Objective(expr=first + operation)
    m.routing = pyo.Constraint(K, inst.V, rule=lambda m, k, i: (
        sum(m.phi[k, a] for a in in_arcs[i]) - sum(m.phi[k, a] for a in out_arcs[i])
        + (m.gam[k, i] if i in inst.S else 0) == ((1 - m.alpha[k]) if i == k else 0)))
    m.rows = [("pipe", e) for e in inst.E] + [("source", s) for s in inst.S] + [("local", k) for k in K]
    R = range(len(m.rows))
    m.lam = pyo.Var(R, domain=pyo.NonNegativeReals)
    m.mu = pyo.Var(R, K, domain=pyo.NonNegativeReals)
    # Add the capacity counterparts from 3a with separate dual variables for each constraint.
    # Keep the demand coefficients and capacity expressions available for 3c.
    return m

*Write your answer here.*

**3c [8 pt] Checking and operating the robust plan.** Work with the $\Gamma=2$ plan and policy. See step 5 of Lecture 9's *Steps for Robust Optimization* and computer lab W5 Exercises 1(c), 2(c) and 4(a).
1. *Capacity check.* Fix $a_k$ at their optimal values and solve the adversarial LP from 3a, $\max_{d\in U(2)}\sum_k d_k a_k$, for each capacity constraint. Skip constraints whose coefficients are all zero. Check residuals with `capacity_report(check_rows)`. Give an optimal dual certificate $(\lambda,\mu)$ for a tight constraint with positive worst-case load, preferably a pipe. If none qualifies, choose a slack constraint with positive load. Check dual feasibility and equality with the LP optimum. What does the check establish, and what error could it miss?
2. *Prescribed policy.* Compute each load $\sum_k d_k(\omega)\,a_k$ at the policy's coefficients in every training winter. Flag winters with a capacity excess above $10^{-6}$ MW and keep the Boolean flags and count for 3d.
3. *Flexible dispatch.* Run `evaluate` on the $\Gamma=2$ plan. From feasibility and 2b's last-resort property, derive how the number of winters with mobile boilers compares with the number of winters with a policy violation, and check this against your counts. Tutorial sheet 5 Exercise 8(b) compares a common robust decision with wait-and-see choices in blending.

In [ ]:
# Reuse the Gamma = 2 model and its routing fractions.
check_rows = []
# Extract the coefficients and capacity of each constraint, solve the adversarial LP
# and append an entry in the form below.
# {"constraint": ..., "worst-case load": ..., "capacity": ...}
# checks = capacity_report(check_rows)
# display(checks)
# print("violations:", int(checks["violated"].sum()))
# Report and verify an optimal dual certificate for one constraint with positive worst-case load.
# Apply the prescribed policy. Save a Boolean flag for each winter and count the violations.
# Compare with the number of winters with mobile boilers from evaluate.

*Write your answer here.*

**3d [7 pt] What does the uncertainty set protect?** A winter $\omega$ is *covered* if some $d\in U(\Gamma)$ is at least as large as its observed demand in every neighborhood. See computer lab W5 Exercise 3(c).
1. *Required budget.* For a training winter $\omega$, consider the LP

   $$
   \min_{\zeta}\ \sum_k\zeta_k
   \quad\text{subject to}\quad
   0\le\zeta_k\le1,\qquad \bar d_k+\hat d_k\zeta_k\ge d_k(\omega)\quad\forall k.
   $$

   Derive its optimal value $\Gamma_{\text{req}}(\omega)$ and explain why it gives the smallest budget that covers the winter. Compute it for all training winters and plot the distribution. For one winter with positive required budget, give an optimal $\zeta$ and check its bounds, dominance inequalities and objective.
2. *Training calibration.* Find $\Gamma^\ast$, the **smallest budget covering at least 90% of the training winters**, with `coverage_budget`. Rounding down can reduce coverage, so keep full precision for the count, the RO solve and `ANSWERS`. Report $\Gamma^\ast$ (rounded only for display), the number covered and the optimal value of RO($\Gamma^\ast$). Keep the plan for 3e.
3. *What coverage guarantees.* From the flags in 3c at $\Gamma=2$, count covered winters with a capacity violation and uncovered winters in which all capacities hold. Prove that coverage implies policy feasibility and discuss the converse using your results, or in general if the policy serves no uncovered winters. Explain how uncertainty-set protection differs from a chance constraint under the 45-scenario distribution.

In [ ]:
def coverage_budget(required):
    # The ceil(0.9*N)-th smallest budget covers at least 90% of the training winters.
    return float(required.sort_values().iloc[math.ceil(0.9 * len(required)) - 1])

In [ ]:
# Write your (commented) code here.

*Write your answer here.*

**3e [6 pt] Advice to the alderman.**

**Table.** With `comparison_table(plan_names)`, compare EV, RP, CC(0.1), RO(2) and RO($\Gamma^\ast$) under flexible dispatch on the 45 training winters. Report first-stage cost, expected annual cost, the number of winters with mobile boilers and 90% CVaR. Mark unavailable plans. Reuse results from 1d, 1e, 2b and 3c and the calibrated plan from 3d, with 1e's scenario-cost and CVaR calculations for the remaining rows.

**Memo.** Recommend a plan in at most 250 words. Base your advice on the cost and count comparison.

1. Give one example of a dominated plan if one exists. A plan is dominated when another has no higher expected cost, no more winters with mobile boilers and a strict improvement on at least one measure. Compare unrounded expected costs with an absolute tolerance of $10^{-3}$ k€/yr and use integer counts.
2. For two adjacent nondominated plans ordered by expected cost and with different counts, compute the extra expected annual cost per avoided winter with mobile boilers. If no such pair exists, explain why.
3. Say whether any plan has lower CVaR than RP and how that affects your advice.
4. Explain how the 45 equally likely winters and the uncertainty sets' guarantees affect your recommendation.
5. Explain why the robust objective from 3b can differ from the expected cost of the same plan under flexible dispatch.

In [ ]:
# Collect the available plans. Reuse earlier evaluations and the scenario costs from 1e where available.
# table = comparison_table(plans)
# Report first-stage cost, mean cost, the number of winters with mobile boilers and 90% CVaR.
# display(table)

*Write your answer here.*

---
## Verification and GenAI use (10 points)

This section is graded mainly on the quality of your verification. Choosing not to use GenAI carries no penalty.

1. **Use of GenAI.** Say whether you used GenAI. If you did, name the tools and give the questions and purposes, such as drafting `build_cc` in 2b or interpreting a solver status in 3d.
2. **Verification.** Cite the cells for checks you already made. For each, say what it showed and which errors it could catch, then explain which errors could pass all your checks. Examples include the one-source check, $\text{WS} \le \text{RP} \le \text{EEV}$, the evaluator audit, the dual certificate and the coverage implication.
3. **One consequential correction or limitation.** Describe a case from your work where GenAI output or your first attempt was wrong, incomplete or misleading. Say how you noticed and what you changed. If nothing went wrong, describe an issue you investigated or an uncertainty assumption you could not verify, and explain why it matters.

1. *...*
2. *...*
3. *...*

---
## Your answers

Enter your own results below. You can reuse values computed earlier, for example `ANSWERS["p1b_cost"] = rp`. Plans are dictionaries in the format returned by `first_stage_of`.

`check_answer_format` checks the required keys and basic types, and `print_answer_block` prints the block staff use for grading. When `ANSWERS` is complete, choose *Restart and run all*, save the notebook with its outputs and upload the `.ipynb` to Canvas.

In [ ]:
ANSWERS_TEMPLATE = {
    "student_number": "id",
    "p1b_cost": "number", "p1b_first_stage": "dict",
    "p1c_r_closed_form": "number", "p1c_r_model": "number",
    "p1d_ws": "number", "p1d_eev": "number", "p1d_evpi": "number", "p1d_vss": "number",
    "p1d_ev_first_stage": "dict",
    "p1e_rp_var": "number", "p1e_rp_cvar": "number",
    "p1e_ev_var": "number", "p1e_ev_cvar": "number",
    "p1f_zero_cost": "number", "p1f_zero_backups": "number",  # audited evaluate, zero plan on training winters
    "p2b_costs": "dict",            # {eps: cost, or None if unavailable}
    "p2b_first_stage": "dict",      # your joint plan for eps = 0.1
    # p2c_max_individual_freq is the largest neighborhood frequency in 2c.2.
    "p2c_max_individual_freq": "number", "p2c_joint_freq": "number", "p2c_eps_ind": "number",
    "p3b_costs": "dict",            # {gamma: cost} for gamma in 0, 1, 2, 3, 4, 6, 8, |K|
    "p3c_first_stage": "dict",      # your robust plan for Gamma = 2
    "p3c_policy_violations_train": "number", "p3c_train_backups": "number",
    "p3d_gamma_req": "dict",        # {winter: smallest budget covering that training winter}
    "p3d_gamma_star": "number", "p3d_ro_gamma_star": "number",
}

In [ ]:
ANSWERS = {key: None for key in ANSWERS_TEMPLATE}
ANSWERS["student_number"] = STUDENT_NUMBER
# ANSWERS["p1b_cost"] = ...

In [ ]:
check_answer_format(ANSWERS, ANSWERS_TEMPLATE)
print_answer_block(ANSWERS)